# 03 — Fit Gaussian Mixture Model

Fit GMM to preprocessed volume with automatic K selection via BIC. Visualize component distributions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from research_ct.io.volume_saver import Load_From_Numpy
from research_ct.segmentation.gmm_fitter import Gmm_Fitter
from research_ct.visualization.plot_distributions import plot_gmm_components

# Load preprocessed volume
Processed = Load_From_Numpy("../data/processed/preprocessed_volume.npz")
print(f"Loaded: {Processed.shape}")

## Flatten Data for GMM

GMM operates on 1D intensity values. Each voxel is treated as an independent sample.

In [ ]:
Flat_Data = Processed.ravel().reshape(-1, 1)
print(f"Total voxels: {Flat_Data.shape[0]:,}")
print(f"Memory: {Flat_Data.nbytes / 1e9:.2f} GB")

# For very large volumes, sample a subset
if Flat_Data.shape[0] > 5_000_000:
    print("Sampling 5M voxels for faster fitting...")
    Indices = np.random.choice(Flat_Data.shape[0], 5_000_000, replace=False)
    Sample_Data = Flat_Data[Indices]
else:
    Sample_Data = Flat_Data

## Fit GMM with BIC Selection

Test K from 2 to 8. BIC penalizes model complexity — the minimum BIC indicates the best trade-off.

In [ ]:
Fitter = Gmm_Fitter(
    Min_Components=2,
    Max_Components=8,
    Covariance_Type="full",
)

Fitter.Fit(Sample_Data, Verbose=True)

## BIC Scores

Plot BIC vs K to visualize the selection.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
K_Range = range(Fitter.Min_Components, Fitter.Max_Components + 1)
ax.plot(K_Range, Fitter.Bic_Scores, 'o-', linewidth=2, markersize=8, color='steelblue')
ax.axvline(Fitter.Num_Components, color='red', linestyle='--', label=f'Selected K={Fitter.Num_Components}')
ax.set_xlabel('Number of Components (K)')
ax.set_ylabel('BIC Score')
ax.set_title('BIC-Based Model Selection')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Component Distributions

Visualize the fitted Gaussian components against the data histogram.

In [ ]:
# Use a smaller sample for plotting (faster)
Plot_Sample = Sample_Data[np.random.choice(len(Sample_Data), 100_000, replace=False)].ravel()

fig = plot_gmm_components(Plot_Sample, Fitter.Model)
plt.show()

## Material Statistics

Extract means, variances, and weights of each component.

In [ ]:
Stats = Fitter.Get_Material_Statistics()

print(f"{'Component':<12} {'Mean':<10} {'Variance':<12} {'Weight':<10}")
print("-" * 50)
for k in range(Fitter.Num_Components):
    print(f"{k:<12} {Stats['Means'][k]:<10.2f} {Stats['Variances'][k]:<12.2f} {Stats['Weights'][k]:<10.4f}")

## Assign Labels to Full Volume

Use the fitted model to classify every voxel.

In [ ]:
# Process in chunks to avoid memory issues
Chunk_Size = 1_000_000
Num_Chunks = int(np.ceil(len(Flat_Data) / Chunk_Size))

Labels = np.empty(len(Flat_Data), dtype=np.int32)
Probs = np.empty((len(Flat_Data), Fitter.Num_Components), dtype=np.float32)

for i in range(Num_Chunks):
    Start = i * Chunk_Size
    End = min((i + 1) * Chunk_Size, len(Flat_Data))
    Chunk = Flat_Data[Start:End]
    
    Labels[Start:End] = Fitter.Predict_Labels(Chunk)
    Probs[Start:End] = Fitter.Predict_Probabilities(Chunk)
    
    if (i + 1) % 10 == 0:
        print(f"Processed {i+1}/{Num_Chunks} chunks")

# Reshape to volume
D, H, W = Processed.shape
Labels_Volume = Labels.reshape(D, H, W)
Probs_Volume = Probs.reshape(D, H, W, Fitter.Num_Components)

print(f"Labels shape: {Labels_Volume.shape}")
print(f"Probabilities shape: {Probs_Volume.shape}")

## Visualize Segmentation Slices

Inspect how labels map to spatial structure.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for i, z in enumerate([D//4, D//2, 3*D//4]):
    # Original
    axes[0, i].imshow(Processed[z], cmap='gray')
    axes[0, i].set_title(f'Processed — Z={z}')
    axes[0, i].axis('off')
    
    # Labels
    axes[1, i].imshow(Labels_Volume[z], cmap='tab10', vmin=0, vmax=Fitter.Num_Components-1)
    axes[1, i].set_title(f'Labels — Z={z}')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Intensity', fontsize=12)
axes[1, 0].set_ylabel('Segmentation', fontsize=12)
plt.suptitle('GMM Segmentation Results', fontsize=14)
plt.tight_layout()
plt.show()

## Save Results

Store labels and probabilities for HMRF and visualization.

In [ ]:
from research_ct.io.volume_saver import Save_As_Numpy

Save_As_Numpy(Labels_Volume.astype(np.uint8), "../data/output/gmm_labels.npz")
Save_As_Numpy(Probs_Volume, "../data/output/gmm_probabilities.npz")

print("Saved labels and probabilities to ../data/output/")